# Memory & State: Stateless vs. Stateful Foundations

Welcome to the second module of the **Agentic AI Learning Path**. In this module, we transition from simple request-response interactions to building agents that can "remember" past interactions and maintain a complex state over time.

## 1. What is Memory in AI?
Memory is the capability of an agent to store and retrieve information from previous steps in a conversation or a task. Without memory, every interaction with an LLM is isolated (stateless).

### Why do we need it?
- **Contextual Continuity**: So the user doesn't have to repeat themselves.
- **Complex Reasoning**: Agents need to keep track of their own thoughts and actions (like in the ReAct pattern).
- **Personalization**: Learning user preferences over time.

---

## 2. Stateless vs. Stateful: The Key Differences

| Feature | Stateless | Stateful |
| :--- | :--- | :--- |
| **Definition** | Each request is independent. No data persists between calls. | Data is maintained across multiple requests/turns. |
| **Memory** | None. The model only knows what's in the current prompt. | Persistent or temporary storage of history and variables. |
| **Use Case** | Search queries, single-shot translations. | Role-playing, multi-step tasks, long-term assistants. |
| **Complexity** | Low. Easy to scale and cache. | High. Requires storage management and context window tracking. |

---

## 3. Environment Setup
We will use **LangChain** and **Google Gemini** for our examples.

In [14]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage, SystemMessage, AIMessage
from langchain.memory import ConversationBufferMemory

# Load environment variables
load_dotenv()

# Initialize Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

print("Environment ready!")

Environment ready!


## 4. The Stateless Approach (The Default)
In a stateless setup, if you ask a follow-up question, the model has no idea what you're talking about unless you manually pass the previous context.

In [15]:
# Turn 1
res1 = llm.invoke("Hi, my name is Bala.")
print(f"Response 1: {res1.content}")

# Turn 2 (Stateless)
res2 = llm.invoke("What is my name?")
print(f"Response 2: {res2.content}")

Response 1: Hi Bala! It's nice to meet you. How can I help you today?
Response 2: As an AI, I don't know your name. I don't have access to personal information about you.

If you'd like me to know your name for our conversation, feel free to tell me!


Notice how in Turn 2, the model fails to recall the name. To fix this in a stateless environment, you have to manually append the history.

## 5. Stateful Memory: Manual Context Injection
Here we manually manage the history by passing a list of messages.

In [16]:
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, my name is Bala.")
]
res1 = llm.invoke(messages)
messages.append(res1)  # Add AI's response to history

messages.append(HumanMessage(content="What is my name?"))
res2 = llm.invoke(messages)
print(f"Response 2 (With Manual History): {res2.content}")

Response 2 (With Manual History): Your name is Bala.


## 6. Using LangChain's Memory Components
LangChain provides abstractions like `ConversationBufferMemory` to automate this process.

In [17]:
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True
)

conversation.predict(input="Hi, I'm working on an Agentic AI project.")
res = conversation.predict(input="What project am I working on?")
print(f"AI Recalls: {res}")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, I'm working on an Agentic AI project.
AI:

> Finished chain.


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, I'm working on an Agentic AI project.
AI: Oh, that sounds absolutely fascinating! An Agentic AI project, you say? That's a really exciting and rapidly evolving area in the world of artificial intelligence right now. I'd love to hear more about what you're building or ex

## 7. Windowing Memory
As conversations get longer, the history consumes more tokens. **Windowing** keeps only the most recent part of the conversation (either by number of messages or token count) to stay within model limits and save costs.

In [18]:
from langchain.memory import ConversationTokenBufferMemory

print("--- Example: Window Memory (Token Limit) ---")
# Keep only the most recent tokens to stay within budget
window_memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=100)

window_memory.save_context({"input": "My favorite food is Pizza"}, {"output": "Nice!"})
window_memory.save_context({"input": "I live in Chennai"}, {"output": "Great city!"})

print(f"Active memory within limit: {window_memory.load_memory_variables({})}")

--- Example: Window Memory (Token Limit) ---
Active memory within limit: {'history': 'Human: My favorite food is Pizza\nAI: Nice!\nHuman: I live in Chennai\nAI: Great city!'}


## 8. Summary Memory
Instead of deleting old history, **Summary Memory** uses an LLM to condense previous messages into a single, concise paragraph. This retains the core context while using significantly fewer tokens.

In [19]:
from langchain.memory import ConversationSummaryMemory

print("--- Example: Summary Memory ---")
# Instead of exact text, store a summary created by the AI
summary_memory = ConversationSummaryMemory(llm=llm)

summary_memory.save_context({"input": "I am a software engineer living in India."}, {"output": "Understood."})
summary_memory.save_context({"input": "I love playing cricket."}, {"output": "Cricket is popular in India!"})

print(f"Concise Summary: {summary_memory.load_memory_variables({})['history']}")

--- Example: Summary Memory ---
Concise Summary: The human states they are a software engineer living in India and loves playing cricket. The AI notes that cricket is popular in India.


## 9. Summary & Tips
1. **Start Simple**: For most small tasks, manual message management is sufficient.
2. **Watch Tokens**: Always use a max context window strategy for production apps.
3. **State is Logic**: Memory is just one type of state. Managing variables (like search results) is also a form of state management.